**主要变量说明**  
| 变量名 | 含义描述 | 来源 | 数据格式 |
|------|----------|------|----------|
| Event | 某次实验完整的事件字典 | pkl 文件读入（来自 `Process.ipynb`） | `dict{event1, event2, ...}` |
| event | Event 中的某一类具体事件 | `Event['keyword']` | `dict{"Time_S": [ts1,ts3,...], "Time_E": [ts2,ts4,...]}` |
| eve_list | Event 中所有事件按时间顺序整理后的列表 | `behave_ana.sort_events()` | `list[(ts1, event), (ts2, event), ...]` |
| trial_sequence | 每一次尝试具体的结果信息 | `behave_ana.transform_sequence()` | `list[(is_correct, correct_side,reward_or_not),...]` |

_*ts指 timestamp_

base内核

In [17]:
import pickle
import behave_ana
import pandas as pd
import os
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans,AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
from scipy.stats import zscore,entropy
from scipy.signal import find_peaks
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


**查找文件**  
找到指定路径下的pkl文件

In [18]:
import re
from pathlib import Path
from datetime import datetime

path = "/mnt/HDD1/zyyk78/"
path = Path(path)
pkl_files = list(path.rglob("*.pkl"))
start = datetime.strptime("20230310", "%Y%m%d")
end = datetime.strptime("20270326", "%Y%m%d")

pkl_files.sort(key=lambda f: re.search(r'(\d{8})', f.name).group(1))

filtered = []
for file in pkl_files:
    match = re.search(r'(\d{8})', file.name)
    if match:
        file_date = datetime.strptime(match.group(1), "%Y%m%d")
        if start <= file_date <= end:
            filtered.append(file)

# for file in filtered:
#     print(file)

# for file in filtered:
#     print(f"    ['{file}'],")


**数据导入和基础统计**  
把pkl路径弄到excel里面,然后把想分析的那一堆copy下就行了

In [19]:
# 自动将上面筛选出的 pkl 路径转成 matches 列表（每个文件单独一个一维列表）
matches = [[str(f)] for f in filtered]
print(matches)
print(f"共 {len(matches)} 个文件")


[['/mnt/HDD1/zyyk78/Stage3/A04/20250311_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A07/20250311_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A06/20250311_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A05/20250311_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A04/20250313_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A07/20250313_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A06/20250313_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A05/20250313_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A04/20250314_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A07/20250314_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A06/20250314_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A05/20250314_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A04/20250317_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A07/20250317_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A06/20250317_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A05/20250317_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A04/20250318_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A07/20250318_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A06/20250318_00.pkl'], ['/mnt/HDD1/zyyk78/Stage3/A05/20250318_00.pkl'], ['/mnt/HDD1/zyyk78/

In [20]:
#new
# 安全取 Event[key]['Time_S']：ModL/ModR 等可能是 numpy.array（历史脏 pkl），不能直接 ['Time_S']
# 对 array 用字符串索引会触发 IndexError: ... are valid indices
def _time_s(Event, key):
    v = Event.get(key)
    if isinstance(v, dict):
        ts = v.get('Time_S')
        return list(ts) if ts is not None else []
    return []  # array / list / None / 缺失 一律当空

data_collect = []
dirty_pkls = []   # 结构异常的 pkl（某 key 不是 dict）
skipped_pkls = [] # 整体跳过的 pkl

for pkl_path in matches:
    try:
        with open(pkl_path[0],"rb") as pkl_f:
            Event=pickle.load(pkl_f)

        # ====== 诊断：检查 Event 各 key 是否为期望的 dict ======
        # 历史 pkl 偶尔会把 ModL/ModR 等存成 numpy.array（而非 {'Time_S':...,'Time_E':...}），
        # 直接 Event['ModL']['Time_S'] 会触发 IndexError: ... are valid indices。
        # sort_events 里对非 dict 的 value 会 continue 跳过，所以 eve_list 里没有 ModL，
        # 但下面第 39 行是硬访问，必须先挡掉。
        need_keys = ['ModL','ModR','LickL','LickR','SlnL','SlnR','ROmL','ROmR']
        bad_keys = [k for k in need_keys if k in Event and not isinstance(Event[k], dict)]
        if bad_keys:
            bad_types = {k: type(Event[k]).__name__ for k in bad_keys}
            print(f'⚠️ 脏 pkl: {pkl_path[0]}  bad_keys={bad_keys}  types={bad_types}')
            dirty_pkls.append({
                'file': pkl_path[0],
                'bad_keys': bad_keys,
                'types': bad_types,
            })
            # ModL/ModR 是 array → 没有正确率可言，直接跳过这个 pkl，不进 data_collect
            # （否则 eve_list 里无 Mod，accuracy_rate 会按"默认 L 边"算出错误但静默的指标）
            skipped_pkls.append(pkl_path[0])
            continue

        long_event=['MinT','Blackhole']
        ignore_list=['MinS','CamT','LickL','LickR','tMod']
        eve_list=behave_ana.sort_events(Event,ignore_list=ignore_list,long_event=long_event)
        date_info=os.path.basename(pkl_path[0])
        date_info=str.split(date_info,'_')[0]
        date_info = date_info[:8]

        tot_rew = tca_auc = log_lr = tot_acc = lrn_auc = avg_licks_after_reward = avg_licks_after_noreward = 0

        try:
            _,_,tot_rew=behave_ana.reward_rate(Event)
            tca_auc = behave_ana.cumulative_attempt_auc(eve_list)
            log_lr = behave_ana.log_lr(eve_list)
        except Exception as e:
            print(f'⚠️ {pkl_path[0]} 指标(reward/auc/lr)失败: {type(e).__name__}: {e}')

        # 关键修复：用 _time_s 安全取值，不再硬 Event['ModL']['Time_S']
        if len(_time_s(Event,'ModL'))==0 and len(_time_s(Event,'ModR'))==0:
            tot_acc=0
        else:
            _,_,tot_acc=behave_ana.accuracy_rate(eve_list)
            trial_sequence = behave_ana.transform_sequence(eve_list)
            lrn_auc = behave_ana.learning_auc(trial_sequence)
            try:
                avg_licks_after_reward, avg_licks_after_noreward, total_lick_events = behave_ana.licking_rate(Event, eve_list)
            except Exception as e:
                print(f'⚠️ {pkl_path[0]} licking_rate 失败: {type(e).__name__}: {e}')
                avg_licks_after_reward = avg_licks_after_noreward = 0

        all_data={
            'pkl_path':pkl_path[0],#原始 pkl 路径（CSV 用，跳过坏 pkl 后仍能对齐）
            'date':date_info,#Date
            'tot_rew':tot_rew,#Reward Ratio
            'tot_acc':tot_acc,#Accuracy
            'lrn_auc' : lrn_auc,#Learning_auc
            'tca_auc' :tca_auc,#Cumulative_Attempt_auc
            'log_lr' : log_lr,#log_lr
            'eve_list':eve_list,
            'avg_licks_after_reward':avg_licks_after_reward,
            'avg_licks_after_noreward':avg_licks_after_noreward
        }

        data_collect.append(all_data)

    except Exception as e:
        print(f'💥 {pkl_path[0]} 整体跳过: {type(e).__name__}: {e}')
        skipped_pkls.append(pkl_path[0])
        continue

# ====== 脏 pkl 汇总 ======
print(f'\n{"=" * 60}')
print(f'脏 pkl 汇总：{len(dirty_pkls)} 个文件含非 dict 的事件 key')
print(f'跳过 pkl：{len(skipped_pkls)} 个（不进入 data_collect）')
print(f'成功 pkl：{len(data_collect)} 个')
print(f'{"=" * 60}')
if dirty_pkls:
    for d in dirty_pkls:
        print(f'  - {d["file"]}')
        print(f'      bad_keys: {d["bad_keys"]}')
        print(f'      types   : {d["types"]}')
    print('\n建议：去源头（fixing.py / ParseEvent.py）查这些 key 是怎么塞进去的，'
          '或者直接重新解析对应 evt 覆盖该 pkl。')
else:
    print('全部 pkl 干净 ✅')

⚠️ /mnt/HDD1/zyyk78/MPtest/B04/20251029_Alm12p9F_00.pkl 指标(reward/auc/lr)失败: ZeroDivisionError: division by zero
⚠️ /mnt/HDD1/zyyk78/MPtest/G02/20251105_T8F_00.pkl 指标(reward/auc/lr)失败: ZeroDivisionError: division by zero
⚠️ /mnt/HDD1/zyyk78/MPtest/G03/20251106_T8F_00.pkl 指标(reward/auc/lr)失败: ZeroDivisionError: division by zero
⚠️ /mnt/HDD1/zyyk78/MPtest/G01/20251107_T8F_00.pkl 指标(reward/auc/lr)失败: ZeroDivisionError: division by zero
⚠️ /mnt/HDD1/zyyk78/MPtest/G04/20251107_T8F_00.pkl 指标(reward/auc/lr)失败: ZeroDivisionError: division by zero
⚠️ /mnt/HDD1/zyyk78/Stage4/D06/20251120_P7F_00.pkl 指标(reward/auc/lr)失败: ZeroDivisionError: division by zero
💥 /mnt/HDD1/zyyk78/Stage4/Miniscope/D04/Cell_trance_21_20260304_153802.pkl 整体跳过: ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
💥 /mnt/HDD1/zyyk78/Stage4/Miniscope/C02/Cell_trance_19_20260318_133932.pkl 整体跳过: ValueError: The truth value of an array with more than one element is ambiguous. 

Date	
Reward Ratio
Accuracy
Cumulative_Attempt_auc
Learning_auc	
log_lr	
avg_licks_after_reward
avg_licks_after_noreward

In [21]:
# #两项简单的统计参数:tot_rew,tot_acc等参数,需要看哪个就打印出来
# for i in ['date', 'tot_rew', 'tot_acc', 'tca_auc', 'lrn_auc', 'log_lr', 'avg_licks_after_reward', 'avg_licks_after_noreward']:
#     print(f"--- {i} ---")
#     for data in data_collect:
#         print(f"{data[i]}")

# #如果报错，可能是同一session有2个以上pkl。酌情删除（）

Switching Latency
Incorrect Switch
Trial_Gap

In [22]:
A_sl=[]#平均Switching Latency
A_err=[]#Incorrect Switch
A_lick_rate=[]#Trial_Gap
for data in data_collect:
    eve_list=data['eve_list']
    lat_data = behave_ana.get_switching_latency(eve_list)
    if len(lat_data) > 0 :
        A_sl.append(sum([i[0] for i in lat_data])/len(lat_data))
        A_err.append(sum([(len(i)-1)/2 for i in lat_data])/len(lat_data))
        lick_event = [ i for i in eve_list if i[1] in ['ROmL','ROmR','SlnL','SlnR'] ]
        A_lick_rate.append( (lick_event[-1][0] - lick_event[0][0]) /  (len(lick_event) * 50000) )
    else:
        A_sl.append(0)
        A_err.append(0)
        A_lick_rate.append(0)

In [23]:
data_collect2= [A_sl,A_err,A_lick_rate]
# for i in data_collect2:
    # for data in i:
        # print(data)
    # print(f"--- ---")


将所有指标保存为CSV表格


In [24]:
# 将所有指标保存为CSV表格
import csv

# 注意：data_collect 可能比 matches 短（跳过了脏 pkl），
# A_sl/A_err/A_lick_rate 是按 data_collect 长度生成的，
# 所以必须按 data_collect 遍历，用同一个 i 取 A_*，不能再用 matches 的 index。
rows = []
for i, data in enumerate(data_collect):
    rows.append({
        'Train_file': data.get('pkl_path', ''),
        'Date': data['date'],
        'Reward Ratio': data['tot_rew'],
        'Accuracy': data['tot_acc'],
        'Cumulative_Attempt_auc': data['tca_auc'],
        'Learning_auc': data['lrn_auc'],
        'log_lr': data['log_lr'],
        'avg_licks_after_reward': data['avg_licks_after_reward'],
        'avg_licks_after_noreward': data['avg_licks_after_noreward'],
        'Switching Latency': A_sl[i],
        'Incorrect Switch': A_err[i],
        'Trial_Gap': A_lick_rate[i],
    })

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_dir = "/home/jyy/Documents/Beh_Ana_J"
csv_path = os.path.join(csv_dir, f"analysis_results_{timestamp}.csv")
fieldnames = [
    'Train_file', 'Date', 'Reward Ratio', 'Accuracy',
    'Cumulative_Attempt_auc', 'Learning_auc', 'log_lr',
    'avg_licks_after_reward', 'avg_licks_after_noreward',
    'Switching Latency', 'Incorrect Switch', 'Trial_Gap'
]
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

print(f"CSV saved to: {csv_path}")
print(f"Total rows written: {len(rows)}")
print(f"（matches={len(matches)}, 成功={len(data_collect)}, 跳过={len(matches)-len(data_collect)}）")

CSV saved to: /home/jyy/Documents/Beh_Ana_J/analysis_results_20260706_163151.csv
Total rows written: 756
（matches=758, 成功=756, 跳过=2）


In [ ]:
# === 诊断：为什么这些指标是 0？===
# 注意：Ev['ModL'] 可能是 array（脏 pkl），用 _time_s 安全取值
def _time_s_d(Event, key):
    v = Event.get(key)
    if isinstance(v, dict):
        ts = v.get('Time_S')
        return list(ts) if ts is not None else []
    return []

diag = []
for pkl_path in matches:
    with open(pkl_path[0], "rb") as f:
        Ev = pickle.load(f)
    modL_empty = (len(_time_s_d(Ev,'ModL')) == 0)
    modR_empty = (len(_time_s_d(Ev,'ModR')) == 0)
    skip_else_branch = modL_empty and modR_empty  # 这是 data_collect 里 if 的命中条件

    # 重跑 get_switching_latency 看为什么返回空
    long_event=['MinT','Blackhole']
    ignore_list=['MinS','CamT','LickL','LickR','tMod']
    eve_list_d = behave_ana.sort_events(Ev, ignore_list=ignore_list, long_event=long_event)
    lat_d = behave_ana.get_switching_latency(eve_list_d)

    # Sln/ROm 这几个事件数量，决定 A_lick_rate 能不能算
    lick_event = [i for i in eve_list_d if i[1] in ['ROmL','ROmR','SlnL','SlnR']]

    diag.append({
        'file': os.path.basename(pkl_path[0]),
        'ModL_empty': modL_empty,
        'ModR_empty': modR_empty,
        'if_branch_triggered(=跳过else)': skip_else_branch,
        'SlnL_n': len(_time_s_d(Ev,'SlnL')),
        'SlnR_n': len(_time_s_d(Ev,'SlnR')),
        'ROmL_n': len(_time_s_d(Ev,'ROmL')),
        'ROmR_n': len(_time_s_d(Ev,'ROmR')),
        'eve_list_total': len(eve_list_d),
        'switch_lat_data_len': len(lat_d),
        'lick_event_n': len(lick_event),
    })

pd.DataFrame(diag)

其他功能

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 从剪贴板读取Excel复制的数据（自动检测分隔符）
df = pd.read_clipboard(header=None)  
# 转换为列表（每行一个子列表）
data_list = df.values.tolist()  
data_0 = data_list[0]
with open(data_0[0],"rb") as pkl_f:
    Event=pickle.load(pkl_f)

fig, (ax1, ax1_5, ax2) = plt.subplots(3, 1, figsize=(20, 5), 
                                   gridspec_kw={'height_ratios': [2 , 1 , 2]},sharex=True,)
sns.rugplot(Event['SlnL']['Time_S'] , ax=ax1, height=0.8, linewidth=0.5, color='green',alpha=0.5)
sns.rugplot(Event['SlnR']['Time_S'] , ax=ax2, height=0.8, linewidth=0.5, color='green',alpha=0.5)
sns.rugplot(Event['ROmL']['Time_S'] , ax=ax1, height=0.8, linewidth=0.5, color='red',alpha=0.5)
sns.rugplot(Event['ROmR']['Time_S'] , ax=ax2, height=0.8, linewidth=0.5, color='red',alpha=0.5)
sns.rugplot(Event['ModL']['Time_S'] , ax=ax1, height=1, linewidth=1.5, color='blue')
sns.rugplot(Event['ModR']['Time_S'] , ax=ax2, height=1, linewidth=1.5, color='blue')

if 'Blackhole' in Event:
    for i in range(len(Event['Blackhole']['Time_S'])):
        ax1_5.fill_betweenx(y=[-0.5, 0.5], x1 = Event['Blackhole']['Time_S'][i], x2=Event['Blackhole']['Time_E'][i],color='red')
ax1.set_yticks([])
ax1.set_ylabel('Left Side')
ax2.set_yticks([])
ax2.set_ylabel('Right Side')
ax1_5.set_yticks([])
ax1_5.set_ylabel('Uncertain Area')
plt.show()

**自回归分析**  
这里只依赖前面载入的pkl文件  
统计这些pkl中选择的历史以来情况

In [ ]:
fig_bys,ax_bys=plt.subplots(figsize=(8, 5),dpi=300)
import numpy as np
para_list=[]
bias_list=[]
acc_list=[]
for i in range(1000):
    sw_seq= []
    ns_seq= []
    for data_dict in data_collect:
        sw=behave_ana.get_switch_sequence(data_dict['eve_list'],10)
        sw_seq.extend(sw)
        ns_seq.extend(behave_ana.get_random_ns_sequence(data_dict['eve_list'],10,len(sw)))
    para,bias,acc=behave_ana.switch_bayes_reg(sw_seq,ns_seq)
    para_list.append(para)
    bias_list.append(bias)
    acc_list.append(acc)
para_list=np.array(para_list)
bias_list=np.array(bias_list)
acc_list=np.array(acc_list)

means = np.mean(para_list, axis=0)[0]
stds = np.std(para_list, axis=0)[0]
means_bias = np.mean(np.squeeze(bias_list))
stds_bias = np.std(np.squeeze(bias_list))
means_acc = np.mean(np.squeeze(acc_list))
stds_acc = np.std(np.squeeze(acc_list))

x = [i for i in range(-means.shape[0],0)]

# 绘制误差线折线图len_lat_data
ax_bys.errorbar(
    [i for i in range(-means.shape[0],0)],   
    y=means, 
    yerr=stds,
    fmt='-o',         # 线型：实线+圆圈标记
    capsize=5,        # 误差线端盖长度
    capthick=2,       # 端盖粗细
    elinewidth=1,     # 误差线粗细
    label='coef ± Std, Expert'
)

ax_bys.errorbar(
    0, 
    y=means_bias, 
    yerr=stds_bias,
    fmt='-o',         # 线型：实线+圆圈标记
    capsize=5,        # 误差线端盖长度
    capthick=2,       # 端盖粗细
    elinewidth=1,     # 误差线粗细
    label='bias ± Std, Expert'
)
ax_bys.axhline(y=0, color='red', linewidth=3, linestyle='-')
print(f'Pred_Acc={means_acc:.2f}±{stds_acc:.2f}')

ax_bys.set_ylabel('Coefficient')
ax_bys.set_xlabel('Trial')
ax_bys.set_title('Logistic Autoregression Coefficient, Bootstrap=1000')
ax_bys.set_ylim(-3, 3)
legend=ax_bys.legend()
display(fig_bys)

**显著性分析比对**   
需要你copy一个表格(直接从excel中框选复制即可)  
参考格式如下:

| Tag | Reward Ratio | Accuracy | Switching Latency | Incorrect Switch | Trial Gap |
|-----|--------------|----------|-------------------|------------------|-----------|
| T8  | 0.534460338  | 0.613312203 | 6.117647059  | 1.441176471  | 3.340758352 |
| T8  | 0.482142857  | 0.18907563  | 16            | 2.357142857  | 2.369864286 |
| T8  | 0.60960961   | 0.599099099 | 9.866666667  | 1.033333333  | 1.945558799 |
| T8  | 0.56097561   | 0.561683599 | 12.73333333  | 0.933333333  | 2.555443079 |
| T9  | 0.489671932  | 0.413122722 | 13            | 1.818181818  | 3.237225881 |
| T9  | 0.363471971  | 0.121661721 | 13.25         | 4.375         | 4.492284458 |
| T9  | 0.520779221  | 0.507792208 | 4.875         | 6.6875        | 3.156573948 |
| T9  | 0.24504084   | 0.175029172 | 5.333333333  | 14.33333333  | 4.780938203 |
  
_并且接下来还有一些要对应修改的_

In [ ]:
ref_tag= 'T9'  #这个是进行显著性分析的参考项
feature_columns = ['Reward Ratio', 'Accuracy', 'Switching Latency', 'Incorrect Switch', 'Trial_Gap']
STAT_TEST = lambda x, y: stats.mannwhitneyu(x, y, alternative="two-sided")  # M-W U-Test
#STAT_TEST = lambda x, y: stats.ttest_ind(x,y)  #T-Test

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

# 从剪贴板读取数据
df = pd.read_clipboard()

# 查看数据结构
print("数据前几行：")
print(df.head())
print("\n数据信息：")
print(df.info())

# 检查数据列名
print("\n数据列名：")
print(df.columns.tolist())


# 数据整理
df_melted = df.melt(id_vars=['Tags'], value_vars=feature_columns, 
                    var_name='Data_Type', value_name='Value')

# 统计检验 - 对每个数据类型，将每个组与ref_tag进行比较
test_results = {}

for data_type in feature_columns:
    data_subset = df_melted[df_melted['Data_Type'] == data_type]
    T9_data = data_subset[data_subset['Tags'] == ref_tag]['Value'].values
    
    # 获取所有唯一的标签
    other_tags = [tag for tag in data_subset['Tags'].unique() if tag != ref_tag]
    
    results = {}
    for tag in other_tags:
        tag_data = data_subset[data_subset['Tags'] == tag]['Value'].values
        if len(T9_data) > 0 and len(tag_data) > 0:
            _, p_value = STAT_TEST(T9_data, tag_data)
            results[tag] = p_value
    
    test_results[data_type] = results

print("\nT检验结果：")
for data_type, tag_results in test_results.items():
    print(f"\n{data_type}:")
    for tag, p_val in tag_results.items():
        print(f"  {tag} vs T9: p-value = {p_val:.4f}")

# 创建图形 - 5个特征，使用2x3的布局
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# 设置颜色 - 使用seaborn默认调色板
unique_tags = df['Tags'].unique()
palette = sns.color_palette("Set2", len(unique_tags))



# 为每个特征绘制箱型图
for i, feature in enumerate(feature_columns):
    if i < len(axes):
        ax = axes[i]
        step_height = 0
        # 绘制箱型图
        sns.boxplot(data=df, x='Tags', y=feature, ax=ax, palette=palette)
        sns.stripplot(data=df, x='Tags', y=feature, ax=ax, color='black', alpha=0.6, size=4, jitter=True)
        
        # 标注显著性
        if feature in test_results:
            y_max = df[feature].max()
            y_min = df[feature].min()
            y_range = y_max - y_min
            
            # 为每个非T9标签添加显著性标注
            x_positions = {}
            for j, tag in enumerate(unique_tags):
                x_positions[tag] = j
            
            T9_position = x_positions[ref_tag]
            
            for tag, p_value in test_results[feature].items():
                if tag in x_positions:
                    tag_position = x_positions[tag]
                    mid_point = (T9_position + tag_position) / 2
                    
                    # 计算标注位置
                    significance_y = y_max + 0.1 * y_range + step_height*0.1*y_range
                    
                    # 添加显著性符号
                    if p_value < 0.001:
                        symbol = '***'
                    elif p_value < 0.01:
                        symbol = '**'
                    elif p_value < 0.05:
                        symbol = '*'
                    else:
                        symbol = 'ns'
                    
                    # 添加横线和标注
                    ax.plot([T9_position, tag_position], [significance_y , significance_y ], 'k-', lw=1)
                    ax.plot([T9_position, T9_position], [significance_y-0.01*y_range , significance_y ], 'k-', lw=1)
                    ax.plot([tag_position, tag_position], [significance_y-0.01*y_range , significance_y ], 'k-', lw=1)
                    ax.text(mid_point, significance_y + 0.02*y_range , symbol, 
                           ha='center', va='bottom', fontsize=10, fontweight='bold')
                step_height += 1      
        
        ax.set_title(f'{feature} Distribution', fontsize=14, fontweight='bold')
        ax.set_ylabel(feature, fontsize=12)
        ax.set_xlabel('Tags', fontsize=12)
        
        # 旋转x轴标签以避免重叠
        ax.tick_params(axis='x', rotation=45)

# 如果子图数量多于特征数量，隐藏多余的子图
for i in range(len(feature_columns), len(axes)):
    axes[i].set_visible(False)

# 调整布局
plt.tight_layout()

# 显示图形
plt.show()

# 显示描述性统计
print("\n描述性统计：")
desc_stats = df.groupby('Tags')[feature_columns].describe()
print(desc_stats)

# 创建显著性结果表格
print("\n显著性总结（与T9比较）:")
significance_summary = []
for feature in feature_columns:
    if feature in test_results:
        for tag, p_val in test_results[feature].items():
            if p_val < 0.001:
                sig_level = '***'
            elif p_val < 0.01:
                sig_level = '**'
            elif p_val < 0.05:
                sig_level = '*'
            else:
                sig_level = 'ns'
            significance_summary.append({
                'Feature': feature,
                'Comparison': f'{tag} vs T9',
                'p-value': f'{p_val:.4f}',
                'Significance': sig_level
            })

significance_df = pd.DataFrame(significance_summary)
print(significance_df)